# Demo 4 — CrewAI: Multiple Specialized Agents Collaborate

Strands (Demo 3) = **one** agent with tools.
**CrewAI** = a *crew* of role-based agents that hand work to each other.

Here: a two-agent content pipeline on Bedrock —
a **Route Analyst** researches (with a tool), then a **Travel Writer**
turns the analysis into a rider briefing. Sequential handoff.

In [5]:
import os
os.environ["OTEL_SDK_DISABLED"] = "true"  # keep demo output clean
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"  # prevent interactive tracing prompt on a TTY

from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool

llm = LLM(model="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0",
          temperature=0.3)

## A tool for the analyst agent

In [6]:
@tool("get_weather")
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    fake_db = {
        "berlin": {"temp_c": 22, "condition": "sunny"},
        "leipzig": {"temp_c": 21, "condition": "cloudy"},
        "nuremberg": {"temp_c": 17, "condition": "rain"},
        "munich": {"temp_c": 18, "condition": "rain"},
    }
    return str(fake_db.get(city.lower(), {"temp_c": 20, "condition": "unknown"}))

## Define the crew: two agents, two tasks

In [7]:
analyst = Agent(
    role="Cycling Route Analyst",
    goal="Assess weather conditions along cycling routes",
    backstory="A meticulous route planner for long-distance cyclists.",
    tools=[get_weather],
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Travel Writer",
    goal="Write short, vivid rider briefings",
    backstory="A cycling journalist who values brevity.",
    llm=llm,
    verbose=True,
)

analyze = Task(
    description=(
        "Check current weather in Berlin, Leipzig, Nuremberg and Munich "
        "(the berlin-munich cycling route). Identify where rain gear is needed."
    ),
    expected_output="A bullet list of cities with conditions and a rain-gear verdict.",
    agent=analyst,
)

brief = Task(
    description="Turn the analysis into a 3-sentence rider briefing.",
    expected_output="A 3-sentence briefing a cyclist reads before departure.",
    agent=writer,
    context=[analyze],
)

crew = Crew(agents=[analyst, writer], tasks=[analyze, brief],
            process=Process.sequential, verbose=True)

## Run the crew

In [8]:
# In Jupyter an asyncio event loop is already running, and newer CrewAI
# versions refuse a synchronous kickoff() there. Running kickoff() in a
# worker thread (which has no event loop) works in every environment —
# notebook or plain terminal, old or new CrewAI.
import asyncio
from concurrent.futures import ThreadPoolExecutor

try:
    asyncio.get_running_loop()  # raises RuntimeError outside Jupyter
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(crew.kickoff).result()
except RuntimeError:
    result = crew.kickoff()

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.16                                                                                       │
│  Latest version:  1.15.18                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1392081d-a9f0-495a-87a0-6ce88c2cb63f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Check current weather in Berlin, Leipzig, Nuremberg and Munich (the berlin-munich cycling route).        │
│  Identify where rain gear is needed.                                                                            │
│  ID: 47d0fc3d-03ba-49d8-acfb-70e50dd5942d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cycling Route Analyst                                                                                   │
│                                                                                                                 │
│  Task: Check current weather in Berlin, Leipzig, Nuremberg and Munich (the berlin-munich cycling route).        │
│  Identify where rain gear is needed.                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_weather                                                                                              │
│  Args: {'city': 'Berlin'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Berlin-Munich Route: Pre-Departure Briefing                                                                  │
│                                                                                                                 │
│  Pack light for Berlin and Leipzig's sunny skies, but stow waterproof gear—rain hits hard from Nuremberg        │
│  south. Temperatures drop from 22°C to 17°C as you roll toward Munich, so layer up and expect wet roads in the  │
│  final stretch. Grab a jacket, rain pants, and fenders before Nuremberg; the weather turns serious there.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Turn the analysis into a 3-sentence rider briefing.                                                      │
│  Agent: Travel Writer                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1392081d-a9f0-495a-87a0-6ce88c2cb63f                                                                       │
│  Final Output: # Berlin-Munich Route: Pre-Departure Briefing                                                    │
│                                                                                                                 │
│  Pack light for Berlin and Leipzig's sunny skies, but stow waterproof gear—rain hits hard from Nuremberg        │
│  south. Temperatures drop from 22°C to 17°C as you roll toward Munich, so layer up and expect wet roads in the  │
│  final stretch. Grab a jacket, rain pants, and fenders before Nuremberg; the weather turns serious there.       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
print(result)

**Berlin-Munich Route Briefing**

Start dry in Berlin and Leipzig with clear skies, but pack rain gear immediately—Nuremberg and Munich are soaked with active precipitation and 17-18°C temps. Waterproof jackets, rain pants, and bike protection are non-negotiable for the southern leg; monitor Leipzig's clouds for early weather shifts. You'll drop 4°C heading south into the rain, so layer up and expect wet roads from Nuremberg onward.


## Takeaways

- CrewAI thinks in **roles, goals, tasks** — orchestration by declaration
- `context=[analyze]` wires task outputs together: agent-to-agent handoff
  *inside one process*
- Strands vs CrewAI isn't either/or: single capable agent vs. a
  division-of-labor pipeline
- Next question: what if the tools live *outside* your process?
  That's **MCP** →